In [1]:
import numpy as np
import os 
import sys
import astropy.units as u
from astropy.io import fits
import matplotlib.pyplot as plt


In [2]:
sys.path.append("/cluster/home/zhuyin/scripts/spice-line-fits/linefit_modules/")

In [3]:
from util import get_mask_errs, get_spice_err
from skew_correction import skew_correct, deskew_linefit_window
from skew_parameter_search import search_shifts, shift_holder, refine_points
from linefit_leastsquares import lsq_fitter, lsq_fitter
from linefit_storage import linefits

fitter = lsq_fitter # lsq_fitter
from linefit_leastsquares import check_for_waves

In [14]:
spice_file = "~/work/spice_psf/spice_data/solo_L2_spice-n-ras_20231031T141032_V22_218104216-000.fits"

In [15]:
with fits.open(spice_file) as hdul:
    hdul.info()
    spice_hdr = hdul[5].header.copy()
    spice_dat = hdul[5].data[0].copy()
spice_dat = spice_dat.transpose([2,1,0]).astype(np.float32)

Filename: /cluster/home/zhuyin/work/spice_psf/spice_data/solo_L2_spice-n-ras_20231031T141032_V22_218104216-000.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  Mg IX 706 - Peak    1 PrimaryHDU     416   (192, 836, 50, 1)   float32   
  1  Ne VIII 770 - Peak    1 ImageHDU       416   (192, 836, 25, 1)   float32   
  2  Ne VIII 780    1 ImageHDU       416   (192, 836, 25, 1)   float32   
  3  S V 786 (Merged)    1 ImageHDU       426   (192, 836, 64, 1)   float32   
  4  C III 977 - Peak    1 ImageHDU       416   (192, 836, 17, 1)   float32   
  5  Ly Beta 1025 (Merged)    1 ImageHDU       423   (192, 836, 31, 1)   float32   
  6  O VI 1032 - Peak    1 ImageHDU       417   (192, 836, 34, 1)   float32   
  7  O VI 1037 (Merged)    1 ImageHDU       426   (192, 836, 62, 1)   float32   
  8  VARIABLE_KEYWORDS    1 BinTableHDU    411   1R x 29C   [192D, 192I, 192I, 192I, 192I, 192I, 192E, 192E, 192E, 192E, 6I, 6I, 6I, 6I, 6J, 6J, 6J, 6J, 4416A, 192D, 192D, 50D, 25D, 25D,

In [6]:
spice_hdr

XTENSION= 'IMAGE   '           / Written by IDL:  Sun Sep  8 18:48:52 2024      
BITPIX  =                  -32 / Real*4 (floating point)                        
NAXIS   =                    4 / Number of dimensions                           
NAXIS1  =                  128 / Number of slit positions (x)                   
NAXIS2  =                  834 / Number of pixels along slit (y)                
NAXIS3  =                   42 / Number of pixels in dispersion dimension       
NAXIS4  =                    1 / Number of exposures per slit position (time)   
PCOUNT  =                    0 /  number of random group parameters             
GCOUNT  =                    1 /  number of random groups                       
DATE    = '2024-09-08T18:48:52' / Date and time of FITS file creation           
                                                                                
EXTNAME = 'C III 977 (Merged)' / Extension name                                 
FILENAME= 'solo_L2_spice-n-r

In [7]:
spice_dx, spice_dy, spice_dl = spice_hdr['CDELT1'],spice_hdr['CDELT2'],10*spice_hdr['CDELT3']
spice_wl0 = 10*spice_hdr['CRVAL3']-spice_dl*spice_hdr['CRPIX3']
spice_la = spice_wl0+spice_dl*np.arange(spice_dat.shape[2],dtype=np.float64)

In [8]:
linelist = {'Ar VIII+S III 700':700.3, 'O III 703':702.9, 'O III 704':703.9, 'Mg IX 706':706.0,
			'O II 718':718.5, 'S IV 745':744.9, 'S IV 748':748.4, 'S IV 750':750.2,
			'O V 759':758.7, 'S IV+O V 759':759.4, 'O V 760':760.3, 'O V 762':762.0,
			'N IV 765':765.1, 'Ne VIII 770':770.4, 'Mg VIII 772':772.3, 'Ne VIII 780':780.3,
			'S V 786':786.5, 'O IV 787':787.7, 'O IV 790':790.1, 'Ly Gamma 972':972.5,
			'C III 977':977.0, 'O I +- Na VI 989':988.7, 'N III 990':989.8, 'N III 992':991.6,
			'H I (+ O I) 1025':1025.7, 'O I 1027':1027.4, 'O VI 1032':1031.9, 'C II 1036':1036.5,
			'O VI 1037':1037.6}

line_names = list(linelist.keys())
line_waves = [linelist[name] for name in line_names]

In [9]:
centers, lines = check_for_waves(spice_la)

In [10]:
xl, xh, yl, yh = [-5,5,-5,5]
xs_initial, ys_initial = np.array(np.meshgrid(np.linspace(xl,xh,5),np.linspace(yl,yh,5))).transpose([0,2,1])

In [11]:
print(xs_initial, ys_initial)

[[-5.  -5.  -5.  -5.  -5. ]
 [-2.5 -2.5 -2.5 -2.5 -2.5]
 [ 0.   0.   0.   0.   0. ]
 [ 2.5  2.5  2.5  2.5  2.5]
 [ 5.   5.   5.   5.   5. ]] [[-5.  -2.5  0.   2.5  5. ]
 [-5.  -2.5  0.   2.5  5. ]
 [-5.  -2.5  0.   2.5  5. ]
 [-5.  -2.5  0.   2.5  5. ]
 [-5.  -2.5  0.   2.5  5. ]]


In [12]:
shift_vars = shift_holder(spice_dat, spice_hdr, fitter.__name__, save_dir='/cluster/home/zhuyin/work/spice_psf/spice_skew_20220330_parallel_test/')

In [13]:
sv_initial = search_shifts(spice_dat, spice_hdr, xs_initial, ys_initial,
                           lsq_fitter, search_multi_thread=True, search_nthread=8,
                           yrange_plot_dir='/cluster/home/zhuyin/work/spice_psf/spice_skew_20220330_parallel_test/yrange_plots/',
                           shift_vars=shift_vars)

/cluster/home/zhuyin/scripts/spice-line-fits/linefit_modules/util.py:92: RuntimeWarning: All-NaN slice encountered
  spice_dat = copy.deepcopy(spice_dat_in) - np.nanmean(np.nanmin(spice_dat_in,axis=0))
/cluster/home/zhuyin/scripts/spice-line-fits/linefit_modules/util.py:94: RuntimeWarning: Mean of empty slice
  signal = np.nanmean(signal_cube,axis=(0,2))
/cluster/home/zhuyin/scripts/spice-line-fits/linefit_modules/util.py:92: RuntimeWarning: All-NaN slice encountered
  spice_dat = copy.deepcopy(spice_dat_in) - np.nanmean(np.nanmin(spice_dat_in,axis=0))
/cluster/home/zhuyin/scripts/spice-line-fits/linefit_modules/util.py:94: RuntimeWarning: Mean of empty slice
  signal = np.nanmean(signal_cube,axis=(0,2))
/cluster/home/zhuyin/scripts/spice-line-fits/linefit_modules/util.py:92: RuntimeWarning: All-NaN slice encountered
  spice_dat = copy.deepcopy(spice_dat_in) - np.nanmean(np.nanmin(spice_dat_in,axis=0))
/cluster/home/zhuyin/scripts/spice-line-fits/linefit_modules/util.py:92: RuntimeWarn

NameError: name 'results' is not defined

In [ ]:
shift_vars.set(sv_initial)

In [ ]:
shift_vars.save()

In [ ]:
x_refine, y_refine = refine_points(shift_vars,[-5,5],[-5,5], 11, 11, 20)

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed